# Module 1 — Introduction and Data Sources

Homework: [cohorts/2026/homework1.md](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework1.md)
Submit: [courses.datatalks.club/sma-zoomcamp-2026/homework/hw01](https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw01)

Kernel: **Python (stock-markets-zoomcamp)** — the shared repo venv.

| Q | Question | Answer |
| --- | --- | --- |
| 1 | Year with most S&P 500 additions since 2020 | **2025** (18 additions) |
| 2 | Indexes beating the S&P 500 YTD (to 2026-08-21) | **2 of 10** |
| 3 | Median drawdown of ≥5% corrections | **7.99%** |
| 4 | Median 2-day return after a positive AMZN surprise | **0.35%** |


## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import io

import numpy as np
import pandas as pd
import requests
import yfinance as yf

from smaz import utils

utils.set_plot_defaults()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

AS_OF = pd.Timestamp("2026-08-21")   # the date the homework anchors on

---

## Question 1 — [Index] S&P 500 stocks added to the index

> Which year had the highest number of additions (starting from 2020)?

Wikipedia blocks the default `pandas` user-agent, so fetch with `requests` first and
hand the HTML to `read_html`.

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
response.raise_for_status()

sp500 = pd.read_html(io.StringIO(response.text))[0]
print(sp500.shape)
sp500.head()

### Build the ticker / name / year-added frame

In [ ]:
additions = sp500[["Symbol", "Security", "Date added"]].rename(
    columns={"Symbol": "ticker", "Security": "name", "Date added": "date_added"}
)
additions["date_added"] = pd.to_datetime(additions["date_added"], errors="coerce")
additions["year_added"] = additions["date_added"].dt.year

print("companies:", len(additions))
print("unparseable dates:", additions["date_added"].isna().sum())
print("range:", additions.date_added.min().date(), "->", additions.date_added.max().date())
additions.head()

### Additions per year since 2020

2026 is excluded from the comparison — it is still in progress, so its count is not a
*full* year and cannot be compared like-for-like.

In [ ]:
per_year = (
    additions[additions.year_added >= 2020]
    .groupby("year_added")
    .size()
    .rename("n_added")
)
print(per_year.to_string())

full_years = per_year[per_year.index < 2026]
print(f"\nHighest full year: {full_years.idxmax()} with {full_years.max()} additions")

In [ ]:
ax = full_years.plot(kind="bar", color="#4C78A8", edgecolor="none")
ax.bar(str(full_years.idxmax()), full_years.max(), color="#E45756")  # highlight the winner
ax.set_title("S&P 500 additions per year (full years, 2020–2025)")
ax.set_xlabel("year added")
ax.set_ylabel("companies added")
for i, v in enumerate(full_years):
    ax.text(i, v + 0.3, str(v), ha="center")

**Answer 1: 2025**, with 18 additions.

> ⚠️ **Caveat.** The Wikipedia table lists only *current* constituents, so this counts
> additions that are **still in the index today**. A company added in 2020 and since
> removed does not appear. Recent years are therefore biased upward relative to older
> ones, and the true number of additions in each year is higher than shown. The ranking
> among 2020–2025 is safe enough, but this is not a clean "additions per year" series.

### Additional: how many current constituents have been in the index more than 20 years?

In [ ]:
tenure_years = (AS_OF - additions["date_added"]).dt.days / 365.25
n_over_20 = int((tenure_years > 20).sum())

print(f"in the index > 20 years: {n_over_20} of {len(additions)} ({n_over_20/len(additions):.1%})")
print(f"i.e. added on or before {(AS_OF - pd.DateOffset(years=20)).date()}")

# 1957-03-04 is the index's own start date, so it is a large single bucket
print("\nadded on 1957-03-04:", int((additions.date_added == "1957-03-04").sum()))

**224 companies** (44.5% of the index) have been constituents for more than 20 years.

---

## Question 2 — [Macro] Indexes YTD (as of 21 August 2026)

> How many indexes (out of 10) have better YTD returns than the S&P 500?

In [ ]:
INDEXES = {
    "^GSPC":     "United States — S&P 500",
    "000001.SS": "China — Shanghai Composite",
    "^HSI":      "Hong Kong — Hang Seng",
    "^AXJO":     "Australia — S&P/ASX 200",
    "^NSEI":     "India — Nifty 50",
    "^GSPTSE":   "Canada — S&P/TSX Composite",
    "^GDAXI":    "Germany — DAX",
    "^FTSE":     "United Kingdom — FTSE 100",
    "^N225":     "Japan — Nikkei 225",
    "^MXX":      "Mexico — IPC",
    "^BVSP":     "Brazil — Ibovespa",
}

# end is exclusive in yfinance, so pass the 22nd to include the 21st
raw = yf.download(
    list(INDEXES), start="2026-01-01", end="2026-08-22",
    auto_adjust=True, progress=False, group_by="ticker",
)
print(raw.shape)

Each market has its own holiday calendar, so the first trading day of 2026 differs by
country. Take each index's own first and last available close rather than forcing a
common date.

In [ ]:
rows = []
for t, label in INDEXES.items():
    s = raw[t]["Close"].dropna()
    rows.append({
        "ticker": t,
        "index": label,
        "first_date": s.index[0].date(),
        "first_close": s.iloc[0],
        "last_date": s.index[-1].date(),
        "last_close": s.iloc[-1],
        "ytd_pct": (s.iloc[-1] / s.iloc[0] - 1) * 100,
    })

ytd = pd.DataFrame(rows).sort_values("ytd_pct", ascending=False).reset_index(drop=True)
ytd.round(2)

In [ ]:
sp500_ytd = ytd.loc[ytd.ticker == "^GSPC", "ytd_pct"].iloc[0]
better = ytd[(ytd.ytd_pct > sp500_ytd) & (ytd.ticker != "^GSPC")]

print(f"S&P 500 YTD: {sp500_ytd:.2f}%")
print(f"indexes beating it: {len(better)} of 10")
for _, r in better.iterrows():
    print(f"  {r['index']:<32} {r.ytd_pct:6.2f}%")

In [ ]:
colors = ["#E45756" if t == "^GSPC" else "#4C78A8" for t in ytd.ticker]
ax = ytd.set_index("index")["ytd_pct"].plot(kind="barh", color=colors, edgecolor="none")
ax.invert_yaxis()
ax.axvline(sp500_ytd, color="#E45756", ls="--", lw=1, label=f"S&P 500 ({sp500_ytd:.1f}%)")
ax.axvline(0, color="#666", lw=0.8)
ax.set_title("YTD performance, 1 Jan – 21 Aug 2026 (local currency, price only)")
ax.set_xlabel("YTD return (%)")
ax.legend()

**Answer 2: 2 of 10** — Japan (Nikkei 225, +27.4%) and Canada (S&P/TSX, +14.9%)
beat the S&P 500's +11.9%.

India is the notable laggard at −7.3%.

### Additional: 3, 5 and 10-year comparisons

In [ ]:
hist = yf.download(
    list(INDEXES), start="2015-01-01", end="2026-08-22",
    auto_adjust=True, progress=False, group_by="ticker",
)

def trailing_growth(s: pd.Series, years: int) -> float:
    """Total % growth over the trailing window, using the last close at or before the start."""
    s = s.dropna()
    if s.index.tz is not None:
        s.index = s.index.tz_localize(None)
    past = s[s.index <= AS_OF - pd.DateOffset(years=years)]
    return float("nan") if past.empty else (s.iloc[-1] / past.iloc[-1] - 1) * 100

perf = pd.DataFrame(
    [
        {
            "index": label,
            "YTD_%": ytd.loc[ytd.ticker == t, "ytd_pct"].iloc[0],
            "3y_%": trailing_growth(hist[t]["Close"], 3),
            "5y_%": trailing_growth(hist[t]["Close"], 5),
            "10y_%": trailing_growth(hist[t]["Close"], 10),
        }
        for t, label in INDEXES.items()
    ]
).set_index("index")

perf.round(1)

In [ ]:
sp_row = perf.loc["United States — S&P 500"]
others = perf.drop("United States — S&P 500")

for col in ["YTD_%", "3y_%", "5y_%", "10y_%"]:
    beat = others[others[col] > sp_row[col]]
    print(f"{col:>6}  S&P {sp_row[col]:7.1f}%  ->  {len(beat)}/10 beat it: "
          f"{', '.join(beat.index) if len(beat) else '—'}")

**Is it the same trend?** Broadly yes, and it gets *stronger* the longer the horizon:

| Horizon | Beat the S&P 500 |
| --- | --- |
| YTD | 2 / 10 |
| 3 years | 2 / 10 |
| 5 years | 2 / 10 |
| 10 years | 1 / 10 |

Only **Japan** beats the US on every horizon (+299% over 10y vs +251%). Canada beats it
on 3 and 5 years but not 10. So the "diversify away from the US" thesis in the linked
article is not supported by price returns alone over the last decade — US
outperformance has been persistent, not just a YTD artifact.

Caveats: local currency (no FX conversion, as the prompt allows), and price-only —
dividends are excluded, which penalises high-yield markets like the FTSE 100 most.

---

## Question 3 — [Index] S&P 500 market corrections

> Calculate the median drawdown (%) of corrections of at least 5%.

In [ ]:
gspc = yf.download("^GSPC", start="1950-01-01", end="2026-08-22",
                   auto_adjust=True, progress=False)
if isinstance(gspc.columns, pd.MultiIndex):
    gspc = gspc.droplevel("Ticker", axis=1)

close = gspc["Close"].dropna()
close.index = pd.to_datetime(close.index)
print(f"{len(close):,} daily closes | {close.index[0].date()} -> {close.index[-1].date()}")

### Step 1 — all-time highs

A day is an all-time high when its close matches the running maximum of every close so far.

In [ ]:
running_max = close.cummax()
is_ath = close >= running_max
ath_dates = close.index[is_ath]

print(f"all-time-high days: {len(ath_dates):,}")

### Steps 2–5 — trough between consecutive highs, drawdown, duration

For each consecutive pair of all-time highs, the trough is the minimum close *strictly
between* them. Pairs of adjacent ATH days have nothing in between and are skipped.

**Duration** is measured peak → trough. That is the definition the prompt's reference
table uses: `2007-10-09 to 2009-03-09 ... 517 days` — 2009-03-09 is the market bottom,
not the recovery date (the index did not regain its 2007 peak until 2013).

In [ ]:
records = []
for d0, d1 in zip(ath_dates[:-1], ath_dates[1:]):
    window = close.loc[d0:d1]
    if len(window) < 3:
        continue  # adjacent highs — no dip in between

    interior = window.iloc[1:-1]
    trough_val = interior.min()
    trough_date = interior.idxmin()
    high = close.loc[d0]

    records.append({
        "high_date": d0,
        "high": high,
        "trough_date": trough_date,
        "trough": trough_val,
        "recovery_date": d1,
        "drawdown_pct": (high - trough_val) / high * 100,
        "duration_days": (trough_date - d0).days,          # peak -> trough
        "recovery_days": (d1 - trough_date).days,          # trough -> new high
    })

drawdowns = pd.DataFrame(records)
print(f"dips between all-time highs: {len(drawdowns):,}")

In [ ]:
corrections = drawdowns[drawdowns.drawdown_pct >= 5.0].reset_index(drop=True)
print(f"corrections of at least 5%: {len(corrections)}")

### Validation against the reference list in the prompt

In [ ]:
top10 = corrections.nlargest(10, "drawdown_pct")
for _, r in top10.iterrows():
    print(f"{r.high_date.date()} to {r.trough_date.date()}: "
          f"{r.drawdown_pct:.1f}% drawdown over {r.duration_days} days")

All ten rows match the prompt's reference list exactly — both the drawdown percentages
and the durations — which confirms the methodology.

### Percentiles

In [ ]:
percentiles = corrections[["drawdown_pct", "duration_days"]].quantile([0.25, 0.5, 0.75])
percentiles.index = ["25th", "50th (median)", "75th"]
percentiles.round(2)

In [ ]:
print(f"median drawdown: {corrections.drawdown_pct.median():.2f}%")
print(f"median duration: {corrections.duration_days.median():.1f} days (peak -> trough)")
print(f"median recovery: {corrections.recovery_days.median():.1f} days (trough -> new high)")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.hist(corrections.drawdown_pct, bins=25, color="#4C78A8", edgecolor="white")
ax1.axvline(corrections.drawdown_pct.median(), color="#E45756", ls="--",
            label=f"median {corrections.drawdown_pct.median():.1f}%")
ax1.set_title("Drawdown depth")
ax1.set_xlabel("drawdown (%)")
ax1.legend()

ax2.hist(corrections.duration_days, bins=25, color="#4C78A8", edgecolor="white")
ax2.axvline(corrections.duration_days.median(), color="#E45756", ls="--",
            label=f"median {corrections.duration_days.median():.0f} days")
ax2.set_title("Duration (peak → trough)")
ax2.set_xlabel("days")
ax2.legend()

fig.suptitle(f"S&P 500 corrections ≥5%, 1950–2026 (n={len(corrections)})")
fig.tight_layout()

**Answer 3: median drawdown = 7.99%** (≈8%).

Supporting numbers, over 74 corrections since 1950:

| Percentile | Drawdown | Duration (peak → trough) |
| --- | --- | --- |
| 25th | 6.23% | 22 days |
| **50th (median)** | **7.99%** | **40.5 days** |
| 75th | 14.02% | 86 days |

The distribution is strongly right-skewed: the typical correction is a shallow ~8% dip
lasting under six weeks, but the tail contains the 2007–09 (56.8%) and dot-com (49.1%)
collapses. Median is the right summary here — the mean would be dragged upward by those
few catastrophes.

---

## Question 4 — [Stocks] Earnings surprise analysis for Amazon

> Median 2-day % change following positive earnings surprise days.

In [ ]:
ticker_obj = yf.Ticker("AMZN")
earnings = ticker_obj.get_earnings_dates(limit=40)

# The prompt's window: 25 entries starting 2020-10-29 (the most recent has no
# reported EPS yet — it is a scheduled future date).
earnings = earnings[earnings.index >= pd.Timestamp("2020-10-29", tz=earnings.index.tz)]
print("entries:", len(earnings))
print("with a reported surprise:", earnings["Surprise(%)"].notna().sum())
earnings.head()

In [ ]:
prices = yf.download("AMZN", start="1997-01-01", end="2026-08-22",
                     auto_adjust=True, progress=False)
if isinstance(prices.columns, pd.MultiIndex):
    prices = prices.droplevel("Ticker", axis=1)

amzn = prices["Close"].dropna()
amzn.index = pd.to_datetime(amzn.index).tz_localize(None)
print(f"{len(amzn):,} closes | {amzn.index[0].date()} -> {amzn.index[-1].date()}")

### 2-day change for every historical date

Day1 = `t-1`, Day2 = `t` (the announcement), Day3 = `t+1`, so the return anchored on the
earnings day is `Close[t+1] / Close[t-1] - 1`.

Computing this for the whole history — not just earnings days — gives the baseline to
compare against. `auto_adjust=True` handles Amazon's 20:1 split in June 2022.

In [ ]:
ret_2d = (amzn.shift(-1) / amzn.shift(1) - 1).rename("ret_2d")

print(f"computed for {ret_2d.notna().sum():,} trading days")
print(f"all-history median 2-day return: {ret_2d.median() * 100:.3f}%")

Amazon reports after the close (16:00 ET), so the announcement-day close is still
*pre*-reaction and the move lands on Day 3 — which is exactly what this window captures.

Earnings dates are aligned forward onto the next available trading day, in case a
report ever lands on a market holiday.

In [ ]:
earnings = earnings.copy()
earnings["earnings_day"] = earnings.index.tz_convert(None).normalize()

trading_days = pd.Series(amzn.index, index=amzn.index)
earnings["trading_day"] = trading_days.reindex(
    earnings["earnings_day"].values, method="bfill"
).values
earnings["ret_2d_pct"] = ret_2d.reindex(earnings["trading_day"].values).values * 100

events = earnings.dropna(subset=["Surprise(%)", "ret_2d_pct"])
print("usable events:", len(events))

In [ ]:
positive = events[events["Surprise(%)"] > 0]
negative = events[events["Surprise(%)"] <= 0]

print(f"positive surprises: {len(positive)}   negative/zero: {len(negative)}\n")
print(positive[["Surprise(%)", "ret_2d_pct"]].sort_index(ascending=False).round(2).to_string())

In [ ]:
print(f"median 2-day return after a POSITIVE surprise: {positive.ret_2d_pct.median():.2f}%")
print(f"  mean:            {positive.ret_2d_pct.mean():.2f}%")
print(f"  share positive:  {(positive.ret_2d_pct > 0).mean():.1%}")
print(f"  range:           {positive.ret_2d_pct.min():.2f}% .. {positive.ret_2d_pct.max():.2f}%")
print(f"\nnegative-surprise median: {negative.ret_2d_pct.median():.2f}%")
print(f"all-history baseline:     {ret_2d.median() * 100:.3f}%")

### Correlation between surprise magnitude and price reaction

In [ ]:
events[["Surprise(%)", "ret_2d_pct"]].corr().round(4)

That +0.22 is not robust. Two events — the 641.9% surprise in Feb 2022 and the 215.0%
in Jul 2026 — sit far outside the rest of the distribution, and Pearson correlation is
dominated by them. Checking with the outliers removed and with a rank-based measure:

In [ ]:
pearson_all = events["Surprise(%)"].corr(events.ret_2d_pct)
spearman_all = events["Surprise(%)"].corr(events.ret_2d_pct, method="spearman")

trimmed = events[events["Surprise(%)"] < 200]
pearson_trimmed = trimmed["Surprise(%)"].corr(trimmed.ret_2d_pct)

print(f"Pearson,  all {len(events)} events        : {pearson_all:+.4f}")
print(f"Pearson,  excluding >200% surprises : {pearson_trimmed:+.4f}   <-- collapses")
print(f"Spearman, all events (rank-based)   : {spearman_all:+.4f}   <-- holds up")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 5))
ax.scatter(events["Surprise(%)"], events.ret_2d_pct,
           c=np.where(events["Surprise(%)"] > 0, "#4C78A8", "#E45756"), s=70, zorder=3)
ax.axhline(0, color="#666", lw=0.8)
ax.axvline(0, color="#666", lw=0.8)
ax.set_xscale("symlog", linthresh=50)
ax.set_xlabel("earnings surprise (%) — symlog scale")
ax.set_ylabel("2-day return (%)")
ax.set_title("AMZN: earnings surprise vs 2-day price reaction (2020-10 – 2026-07)")

for _, r in events.iterrows():
    if abs(r["Surprise(%)"]) > 200 or abs(r.ret_2d_pct) > 9:
        ax.annotate(str(r.trading_day.date()), (r["Surprise(%)"], r.ret_2d_pct),
                    textcoords="offset points", xytext=(6, 4), fontsize=8, color="#444")

**Answer 4: median 2-day return after a positive surprise = 0.35%.**

Correlation with surprise magnitude: **+0.22** (Pearson, all 24 events).

What the numbers actually say:

- 0.35% is barely above the **0.16%** all-history baseline for any random 2-day window,
  and only **55%** of positive surprises produced a positive move — close to a coin flip.
- The spread is enormous: −10.6% (Oct 2022) to +19.8% (Jul 2026). The median hides that.
- The strongest counter-example is 2026-02-05: a surprise of +0.22% — technically
  positive, but effectively an in-line print — was met with **−9.73%**.
- The +0.22 Pearson correlation **falls to −0.04** once the two >200% surprises are
  excluded, while rank-based Spearman stays at **+0.28**. So there is a weak *monotonic*
  tendency for bigger beats to be rewarded, but no reliable linear relationship.

The practical reading: beating consensus is not itself tradeable. The market prices in
an expected beat — Amazon beat in 20 of 24 quarters — so what moves the stock is the
*size* of the beat relative to that expectation, plus guidance, which is not in this
dataset at all. `n=24` is also far too small to conclude much with confidence.

**Bull vs bear markets** (the second follow-up) would need a regime label — e.g.
tagging each event by whether the S&P 500 was above its 200-day moving average — and
with only ~24 events, splitting further leaves too few per bucket to be meaningful.

---

## Summary of answers

| Q | Question | Answer |
| --- | --- | --- |
| 1 | Year with most S&P 500 additions since 2020 | **2025** (18 additions) |
| 1b | Current constituents in the index >20 years | **224** (44.5%) |
| 2 | Indexes beating the S&P 500 YTD | **2 of 10** (Japan, Canada) |
| 2b | Beating it over 3 / 5 / 10 years | **2 / 2 / 1** — only Japan on all horizons |
| 3 | Median drawdown of ≥5% corrections | **7.99%** |
| 3b | Median duration (peak → trough) | **40.5 days** |
| 4 | Median 2-day return after a positive surprise | **0.35%** |
| 4b | Correlation with surprise magnitude | **+0.22** (Pearson) — not robust; +0.28 Spearman |

Data as of **2026-08-21**.